# Pneumonia-X — étape 2 : entraînement sur GPU Kaggle

Avant de lancer (panneau de droite, *Session options*) :
1. **Accelerator : GPU T4 x2** (ou P100).
2. **Internet : On**.
3. **Add Input** : chercher `chest-xray-pneumonia` (auteur *paultimothymooney*) et l'ajouter.

Puis *Run All*. Durée : environ 30 à 45 min pour les 3 modèles.

In [ ]:
!nvidia-smi -L
import glob

print(glob.glob("/kaggle/input/*"))

In [ ]:
%cd /kaggle/working
!rm -rf pneumonia && git clone -q https://github.com/khadijafair/pneumonia.git
%cd /kaggle/working/pneumonia
!git log --oneline -1
%env PYTHONPATH=/kaggle/working/pneumonia/src
%env MLFLOW_DISABLE_AGENT_HINT=1
!pip install -q -r requirements-train.txt

## Comparaison de 3 architectures (même découpage par patient, même graine)

In [ ]:
DATA = glob.glob("/kaggle/input/*chest-xray-pneumonia*")[0]
print(DATA)
!python -m pneumonia.train -m model=efficientnet_b0,densenet121,vit_small data.root={DATA}

In [ ]:
import json, pandas as pd

rows = [json.load(open(p)) for p in sorted(glob.glob("outputs/*/summary.json"))]
pd.DataFrame(rows).set_index("model")

## Vérification des identifiants patients avec des embeddings profonds

La mesure doit d'abord séparer B (même patient) de C (patients différents).

In [ ]:
!python scripts/audit_patient_ids.py --features deep --root {DATA}
!python scripts/audit_patient_ids.py --features deep --root {DATA} --ckpt outputs/densenet121/best.ckpt

## Résultats à télécharger (Output → `results.zip`)

In [ ]:
import shutil, pathlib

res = pathlib.Path("results")
shutil.rmtree(res, ignore_errors=True)
res.mkdir()
for d in pathlib.Path("outputs").iterdir():
    if (d / "summary.json").exists():
        (res / d.name).mkdir()
        for f in [d / "summary.json", *d.glob("pred_*.csv")]:
            shutil.copy(f, res / d.name)
for f in pathlib.Path("data/splits").glob("patient_id_audit_deep*.md"):
    shutil.copy(f, res)
!cd /kaggle/working/pneumonia && zip -qr /kaggle/working/results.zip results mlflow.db mlartifacts
!ls -la /kaggle/working/results.zip && find results -type f | sort